In [1]:
import json
import time
import requests

# Link to the PokeAPI to get all the pokemon and items
POKEAPI = "https://pokeapi.co/api/v2"

In [2]:
# Functions

# Get the json data from a specific URL
def FetchJSON(url, cache):
    if url in cache:
        return cache[url]
    
    for _ in range(3):
        try:
            data = requests.get(url, timeout=30).json()
            cache[url] = data
            return data
        except Exception:
            time.sleep(0.5)
    raise RuntimeError(f"Failed Request: {url}")

# Get the moves for each specific pokemon
def GetMoves(move):
    return {
        "name": move["name"],
        "type": move["type"]["name"] if move.get("type") else "",
        "category": move["damage_class"]["name"] if move.get("damage_class") else "status",
        "power": move.get("power") or 0,
        "accuracy": move.get("accuracy") or 0,
        "pp": move.get("pp") or 0,
        "priority": move.get("priority") or 0
    }

# Get pokedex type and number
def GetPokedexNumber(species, pokedexName):
    for entry in species.get("pokedex_numbers", []):
        pokedex = entry.get("pokedex", {})
        if pokedex.get("name") == pokedexName:
            return entry.get("entry_number")
    return None

In [3]:
# Get the pokemon data

pokemonData = []            # Store the data of the pokemon
requestCache = {}           # Storage container to store the cache recieved when getting the JSON
moveCache = {}              # Storage container to store the cache recieve when the getting the moves JSON


pokemonList = FetchJSON(f"{POKEAPI}/pokemon?limit=10000", requestCache)["results"]

for entry in pokemonList:
    
    # Get the specific url within the pokeapi to get the pokmon
    pokemonURL = entry["url"]
    pokemon = FetchJSON(pokemonURL, requestCache)
    
    # Get the pokemon itself
    speciesURL = pokemon["species"]["url"]
    species = FetchJSON(speciesURL, requestCache)
    
    # Get the form of the pokemon to exclude. Final list needs to not include forms as they are bad data
    formURL = f"{POKEAPI}/pokemon-form/{pokemon['name']}"
    try:
        formData = FetchJSON(formURL, requestCache)
    except Exception:
        formData = {
            "name": pokemon["name"],
            "form_name": "",
            "is_default": pokemon.get("is_default", False),
            "is_battle_only": False,
            "is_mega": False,
            "form_order": 1,
            "order": pokemon.get("order", 0)
        }
    
    # Get the gender and generation
    gender = species.get("gender_rate", -1)
    generation = species["generation"]["name"] if species.get("generation") else ""
    
    # Get the pokedex type and number
    paldeaDex = GetPokedexNumber(species, "paldea")
    kitakamiDex = GetPokedexNumber(species, "kitakami")
    blueberryDex = GetPokedexNumber(species, "blueberry")
    
    # Get the base stat total
    stats = {s["stat"]["name"]: s["base_stat"] for s in pokemon["stats"]}
    bst = sum(stats.values())
    
    # Start of getting the moves
    moveDetails = {}
    moveNames = []
    
    # Get the name and URL to start
    for moveEntry  in pokemon.get("moves", []):
        moveName = moveEntry["move"]["name"]
        moveURL = moveEntry["move"]["url"]
        moveNames.append(moveName)
        
        # Get the metadata of the moves
        try:
            moveData = FetchJSON(moveURL, moveCache)
            moveDetails[moveName] = GetMoves(moveData)
        except Exception:
            moveDetails[moveName] = {
                "name": moveName,
                "type": "",
                "category": "status",
                "power": 0,
                "accuracy": 0,
                "pp": 0,
                "priority": 0,
            }
    
    # Append the pokemon and its metadata to the list to write out to json
    pokemonData.append({
        "name": pokemon["name"],
        "species_name": species["name"],
        "form_name": formData.get("form_name", ""),
        "is_default_form": formData.get("is_default", pokemon.get("is_default", False)),
        "is_battle_only": formData.get("is_battle_only", False),
        "is_mega": formData.get("is_mega", False),
        "form_order": formData.get("form_order", 0),
        "order": formData.get("order", pokemon.get("order", 0)),
        "generation": generation,
        "type": [t["type"]["name"] for t in pokemon["types"]],
        "stats": stats,
        "bst": bst,
        "abilities": [
            {
                "name": a["ability"]["name"],
                "hidden": a["is_hidden"]
            }
            for a in pokemon["abilities"]
        ],
        "moveset": sorted(set(moveNames)),
        "move_details": moveDetails,
        "gender_rate": gender,
        "paldea_dex": paldeaDex,
        "kitakami_dex": kitakamiDex,
        "blueberry_dex": blueberryDex,
    })
    
    time.sleep(0.01)
    
with open("../../data/pokemon_data.json", "w", encoding="utf-8") as f:
    json.dump(pokemonData, f, indent=2)
    
print(f"Saved {len(pokemonData)}")

Saved 1350


In [4]:
# Start of getting the items

items = []
itemsCache = {}

# Get the URL
data = FetchJSON(f"{POKEAPI}/item?limit=10000", itemsCache)

# Get the allowed categories to exclude bad items from the list but still include all legal items
ALLOWEDCATEGORIES = {
    "held-items",
    "berries",
    "choice",
    "training",
    "plates",
    "species-specific",
    "type-protection",
    "jewels",
    "vitamins",
    "loot",
    "healing",
    "pp-recovery",
    "status-cures"
}

# Some items to force include
COMPETITIVEITEMS = {
    "sitrus-berry",
    "assault-vest",
    "rocky-helmet",
    "choice-specs",
    "choice-band",
    "choice-scarf",
    "focus-sash",
    "mystic-water",
    "clear-amulet",
    "covert-cloak",
    "safety-goggles",
    "loaded-dice",
    "weakness-policy",
    "booster-energy",
    "mirror-herb",
    "eject-button",
    "eject-pack",
    "protective-pads",
    "punching-glove",
    "white-herb"
}

# Get item data and append to the list to write out to json
for itemEntry in data["results"]:
    itemData = FetchJSON(itemEntry["url"], itemsCache)
    
    category = itemData.get("category", {}).get("name", "")
    attributes = {a["name"] for a in itemData.get("attributes", [])}
    
    if category in ALLOWEDCATEGORIES or "holdable" in attributes:
        items.append(itemEntry["name"])

items = sorted(set(items)| COMPETITIVEITEMS)

with open("../../data/items.json", "w", encoding="utf-8") as f:
    json.dump(items, f, indent=2)